# 📖 Notebook 2: Pagination Strategies

## Why Pagination Matters

Imagine you have a database with **1 million events**. If your API returned all of them
in a single response, that response could be **hundreds of megabytes**. The server would
run out of memory, the network would choke, and the client would crash trying to parse
all that data.

**Pagination** solves this by breaking large result sets into small, manageable **pages**.
Instead of asking "give me everything", you ask "give me the next 10 items".

Think of it like Google search results — when you search something, Google doesn't show
you all 1,000,000 results on one page. It shows you page 1 with 10 results, and you
click "Next" to see more.

## 🎯 Learning Objectives

By the end of this notebook you will:

1. Understand **why** every API should paginate its responses
2. Know how **offset-based** pagination works and when to use it
3. Know how **cursor-based** pagination works and when to use it
4. Understand the **trade-offs** between both approaches
5. Be able to discuss pagination strategies in a system-design interview

## ⚙️ Setup

Make sure the lab environment is running:

```bash
cd core-concepts/api-design
docker-compose up -d
```

### Services

| Service   | URL                                    | Purpose                      |
|-----------|----------------------------------------|------------------------------|
| FastAPI   | http://localhost:8000/docs              | Interactive API docs (Swagger) |
| Adminer   | http://localhost:8080                   | Visual database browser      |
| PostgreSQL| localhost:5432                          | Database (api_design_demo)   |

### Kernel Selection

This notebook needs the local `.venv` Python kernel:

1. Create it (once): `uv sync`
2. In VS Code, click the kernel picker in the **top-right** of this notebook
3. Select **`.venv (Python 3.x)`**
4. If the kernel doesn't appear, reload the window: `Cmd+Shift+P` → *"Reload Window"*

In [1]:
# ── Imports & Configuration ──────────────────────────────────────────────────
import requests   # for calling our API
import psycopg2   # for direct PostgreSQL queries
import json       # for pretty-printing JSON
import time       # for measuring performance
import math       # for page calculations

# Our FastAPI server address
BASE_URL = "http://localhost:8000"

# Direct database connection settings
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "api_design_demo",
    "user": "demo",
    "password": "demo",
}


def pretty(data):
    """Pretty-print a dict / JSON response so it's easy to read."""
    print(json.dumps(data, indent=2, default=str))


def api_get(path, **params):
    """Helper: call GET on our API and return the parsed JSON."""
    resp = requests.get(f"{BASE_URL}{path}", params=params)
    resp.raise_for_status()
    return resp.json()


# ── Quick connection test ─────────────────────────────────────────────────────
health = requests.get(f"{BASE_URL}/health").json()
print("✅ API is running:", health)

conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()
cur.execute("SELECT count(*) FROM events")
print(f"✅ Database connected — {cur.fetchone()[0]} events in the database")
cur.close()
conn.close()

✅ API is running: {'status': 'ok'}
✅ Database connected — 52 events in the database


## 🤔 Why Pagination?

Let's see what happens when we try to return **all** events at once.

Right now we only have 50 events — that's fine. But what if we had **1 million**?

### The Analogy

Think about ordering food at a restaurant:
- **Without pagination**: "Bring me EVERYTHING on the menu at once!" 🍽️🍽️🍽️ → the table collapses
- **With pagination**: "Bring me the appetizers first, then the main course" → manageable

The same applies to APIs. Let's see the numbers:

In [2]:
# ── How big would an unpaginated response be? ────────────────────────────────

# Count total events
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()
cur.execute("SELECT count(*) FROM events")
total_events = cur.fetchone()[0]

# Fetch ONE event to estimate the size of each record
cur.execute("SELECT * FROM events LIMIT 1")
one_row = cur.fetchone()
cur.close()
conn.close()

# Estimate the size of a single event as JSON (rough approximation)
one_event_size = len(json.dumps(one_row, default=str).encode("utf-8"))

print(f"📊 Total events in database: {total_events}")
print(f"📦 Approximate size of 1 event: {one_event_size} bytes")
print()
print("What if we had more events?")
print("─" * 50)

# Show how response size grows
for count in [50, 1_000, 100_000, 1_000_000]:
    size_bytes = count * one_event_size
    if size_bytes < 1024:
        size_str = f"{size_bytes} B"
    elif size_bytes < 1024 * 1024:
        size_str = f"{size_bytes / 1024:.1f} KB"
    else:
        size_str = f"{size_bytes / (1024 * 1024):.1f} MB"
    print(f"  {count:>10,} events → ~{size_str}")

print()
print("💡 At 1M events the response would be hundreds of MB!")
print("   That's why we ALWAYS paginate API responses.")

📊 Total events in database: 52
📦 Approximate size of 1 event: 206 bytes

What if we had more events?
──────────────────────────────────────────────────
          50 events → ~10.1 KB
       1,000 events → ~201.2 KB
     100,000 events → ~19.6 MB
   1,000,000 events → ~196.5 MB

💡 At 1M events the response would be hundreds of MB!
   That's why we ALWAYS paginate API responses.


---

## 📄 Strategy 1: Offset-Based Pagination

### How It Works

Offset-based pagination uses two parameters:
- **`offset`** — how many records to skip
- **`limit`** — how many records to return

It's like telling the database:

> "Skip the first 20 records and give me the next 10."

### SQL Equivalent

```sql
SELECT * FROM events ORDER BY id OFFSET 20 LIMIT 10;
```

### Page-to-Offset Mapping

| Page | Offset | Limit | Records Returned |
|------|--------|-------|------------------|
| 1    | 0      | 10    | 1 – 10           |
| 2    | 10     | 10    | 11 – 20          |
| 3    | 20     | 10    | 21 – 30          |
| 5    | 40     | 10    | 41 – 50          |

The formula is: **`offset = (page - 1) × limit`**

### ✅ Advantages
- Simple and intuitive
- Supports **"jump to page N"** (random access)
- Easy to build a page-number navigation bar

### Our API Endpoint

```
GET /v1/events?offset=0&limit=10&category=music
```

In [3]:
# ── Offset Pagination in Action ──────────────────────────────────────────────
# Let's page through our events 5 at a time using the V1 API.

PAGE_SIZE = 5  # how many events per page

for page_num in range(1, 4):  # pages 1, 2, 3
    # Calculate offset from page number
    offset = (page_num - 1) * PAGE_SIZE

    # Call the API
    data = api_get("/v1/events", offset=offset, limit=PAGE_SIZE)

    # Print results
    print(f"\n{'═' * 60}")
    print(f"📄 PAGE {page_num}  (offset={offset}, limit={PAGE_SIZE})")
    print(f"{'═' * 60}")

    for event in data["events"]:
        print(f"  #{event['id']:>3}  {event['title'][:45]:<45}  [{event['category']}]")

    # Show pagination metadata returned by the API
    pag = data["pagination"]
    total_pages = math.ceil(pag["total"] / pag["limit"])
    print(f"\n  Pagination: offset={pag['offset']}, limit={pag['limit']}, total={pag['total']}")
    print(f"  Page {page_num} of {total_pages}")


════════════════════════════════════════════════════════════
📄 PAGE 1  (offset=0, limit=5)
════════════════════════════════════════════════════════════
  #  1  Jazz Night Vol. 1                              [tech]
  #  2  Comedy Show: Laughs Unlimited 2                [tech]
  #  3  Tech Conference 2024                           [sports]
  #  4  Classical Concert Series 4                     [tech]
  #  5  Rock Festival 2026 #5                          [sports]

  Pagination: offset=0, limit=5, total=52
  Page 1 of 11

════════════════════════════════════════════════════════════
📄 PAGE 2  (offset=5, limit=5)
════════════════════════════════════════════════════════════
  #  6  Jazz Night Vol. 6                              [sports]
  #  7  Comedy Show: Laughs Unlimited 7                [arts]
  #  8  Tech Conference 2026                           [comedy]
  #  9  Classical Concert Series 9                     [music]
  # 10  Rock Festival 2025 #10                         [arts]

  Pagi

In [4]:
# ── Calculating Total Pages ──────────────────────────────────────────────────
# A common pattern: figure out how many pages exist so you can build navigation.

# Fetch the first page just to get the total count
first_page = api_get("/v1/events", offset=0, limit=10)
total = first_page["pagination"]["total"]
page_size = 10

total_pages = math.ceil(total / page_size)

print(f"Total events: {total}")
print(f"Page size:    {page_size}")
print(f"Total pages:  {total_pages}")
print()
print("Page navigation bar:")
print("  ", end="")
for p in range(1, total_pages + 1):
    print(f" [{p}] ", end="")
print("\n")
print("💡 This is what you see at the bottom of most websites!")

Total events: 52
Page size:    10
Total pages:  6

Page navigation bar:
   [1]  [2]  [3]  [4]  [5]  [6] 

💡 This is what you see at the bottom of most websites!


## ⚠️ Problems With Offset Pagination

Offset pagination is simple, but it has two big problems:

### Problem 1: Performance Degrades With Large Offsets

When you say `OFFSET 1,000,000`, the database doesn't magically jump to row 1,000,000.
It actually **scans through all 1,000,000 rows** and then throws them away! This gets
slower and slower as the offset grows.

### Problem 2: Unstable Results (The "Shifting Window")

If someone inserts a new record while you're paginating, the results shift. You might
**see the same record twice** on different pages, or **miss a record entirely**.

Let's demonstrate both problems:

In [5]:
# ── Problem 1: Large offsets are slow ────────────────────────────────────────
# Let's look at the query plan to see what happens with different offsets.
# (With only 50 rows the time difference is tiny, but the COST tells the story.)

conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

print("🔍 EXPLAIN ANALYZE: How the database handles different offsets\n")

for offset_val in [0, 10, 40]:
    cur.execute(
        f"EXPLAIN ANALYZE SELECT * FROM events ORDER BY id OFFSET {offset_val} LIMIT 5"
    )
    plan = cur.fetchall()
    print(f"── OFFSET {offset_val}, LIMIT 5 ──")
    for row in plan:
        print(f"  {row[0]}")
    print()

cur.close()
conn.close()

print("💡 Notice: even though we only want 5 rows, the database scans more")
print("   rows as the offset increases. With millions of rows, this is a disaster!")

🔍 EXPLAIN ANALYZE: How the database handles different offsets

── OFFSET 0, LIMIT 5 ──
  Limit  (cost=0.14..1.49 rows=5 width=133) (actual time=0.005..0.006 rows=5 loops=1)
    ->  Index Scan using events_pkey on events  (cost=0.14..13.91 rows=51 width=133) (actual time=0.004..0.005 rows=5 loops=1)
  Planning Time: 0.218 ms
  Execution Time: 0.028 ms

── OFFSET 10, LIMIT 5 ──
  Limit  (cost=3.79..3.80 rows=5 width=133) (actual time=0.026..0.027 rows=5 loops=1)
    ->  Sort  (cost=3.76..3.89 rows=51 width=133) (actual time=0.026..0.026 rows=15 loops=1)
          Sort Key: id
          Sort Method: top-N heapsort  Memory: 28kB
          ->  Seq Scan on events  (cost=0.00..2.51 rows=51 width=133) (actual time=0.002..0.010 rows=52 loops=1)
  Planning Time: 0.022 ms
  Execution Time: 0.035 ms

── OFFSET 40, LIMIT 5 ──
  Limit  (cost=4.06..4.07 rows=5 width=133) (actual time=0.011..0.012 rows=5 loops=1)
    ->  Sort  (cost=3.96..4.08 rows=51 width=133) (actual time=0.009..0.011 rows=45 loops

In [6]:
# ── Problem 2: Unstable results when data changes ────────────────────────────
# This simulates what happens when a new event is inserted between page fetches.

print("🔄 Simulating the 'shifting window' problem\n")

# Step 1: Fetch page 1 (events 1-5)
page1 = api_get("/v1/events", offset=0, limit=5)
page1_ids = [e["id"] for e in page1["events"]]
print(f"Page 1 event IDs: {page1_ids}")

# Step 2: Insert a new event (simulating another user creating data)
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()
cur.execute(
    """INSERT INTO events (title, description, venue_id, event_date, price, total_tickets, category)
       VALUES ('Surprise Event!', 'Inserted mid-pagination', 1, NOW(), 50.00, 100, 'music')
       RETURNING id"""
)
new_id = cur.fetchone()[0]
conn.commit()
print(f"\n⚡ New event inserted with id={new_id}")
print(f"   Total is now {page1['pagination']['total'] + 1}")

# Step 3: Fetch page 2 — but wait, the total has shifted!
page2 = api_get("/v1/events", offset=5, limit=5)
page2_ids = [e["id"] for e in page2["events"]]
print(f"\nPage 2 event IDs: {page2_ids}")

# Check for overlap
overlap = set(page1_ids) & set(page2_ids)
if overlap:
    print(f"\n⚠️  DUPLICATE! Event(s) {overlap} appeared on BOTH pages!")
else:
    print("\n   (No duplicates in this run — but the total shifted, which can")
    print("    cause issues on later pages or with total page count.)")

print(f"\n   Page 1 saw total={page1['pagination']['total']}, "
      f"Page 2 sees total={page2['pagination']['total']}")
print("   The ground shifted under our feet! 🌊")

# Cleanup: remove the inserted event
cur.execute("DELETE FROM events WHERE id = %s", (new_id,))
conn.commit()
cur.close()
conn.close()
print(f"\n🧹 Cleaned up: deleted event id={new_id}")

🔄 Simulating the 'shifting window' problem

Page 1 event IDs: [1, 2, 3, 4, 5]

⚡ New event inserted with id=62
   Total is now 53

Page 2 event IDs: [6, 7, 8, 9, 10]

   (No duplicates in this run — but the total shifted, which can
    cause issues on later pages or with total page count.)

   Page 1 saw total=52, Page 2 sees total=53
   The ground shifted under our feet! 🌊

🧹 Cleaned up: deleted event id=62


---

## 🔖 Strategy 2: Cursor-Based Pagination

### How It Works

Instead of saying "skip N rows", cursor pagination says:

> "Give me the next 10 records **after this specific record**."

The **cursor** is like a **bookmark** — it marks where you left off. Typically the
cursor is the **ID** of the last record you received.

### SQL Equivalent

```sql
-- Instead of: SELECT * FROM events OFFSET 20 LIMIT 10
-- We do:      SELECT * FROM events WHERE id > 20 ORDER BY id LIMIT 10
```

The key difference: `WHERE id > 20` uses the **index** to jump directly to the right
row, instead of scanning through rows 1–20 first.

### How the Flow Works

```
Client                              Server
  │                                    │
  │── GET /events?limit=5 ────────────→│   (no cursor = start from beginning)
  │←── {events: [...], cursor: 5} ─────│
  │                                    │
  │── GET /events?cursor=5&limit=5 ───→│   ("give me items after id=5")
  │←── {events: [...], cursor: 10} ────│
  │                                    │
  │── GET /events?cursor=10&limit=5 ──→│   ("give me items after id=10")
  │←── {events: [...], has_more: false}│   (last page!)
```

### ✅ Advantages
- **Fast at any position** — uses index, no full scan
- **Stable** — inserts don't cause duplicates or missed records
- Perfect for **infinite scroll** (mobile apps, social media feeds)

### ❌ Disadvantages
- **No random page access** — can't jump to "page 5"
- Slightly more complex to implement

### Our API Endpoint

```
GET /v2/events?cursor=5&limit=10&category=music
```

In [7]:
# ── Cursor Pagination in Action ──────────────────────────────────────────────
# Let's page through our events using the V2 (cursor-based) API.
#
# The pattern:
#   1. First request: no cursor (start from the beginning)
#   2. Each response includes a `next_cursor`
#   3. Pass that cursor to the next request
#   4. Stop when `has_more` is False

PAGE_SIZE = 5
cursor = None       # None means "start from the beginning"
page_num = 0

print("🔖 Paginating with cursors (3 pages)\n")

for _ in range(3):  # fetch 3 pages
    page_num += 1

    # Build the request parameters
    params = {"limit": PAGE_SIZE}
    if cursor is not None:
        params["cursor"] = cursor   # "give me items after this id"

    # Call the API
    data = api_get("/v2/events", **params)

    # Print results
    print(f"{'═' * 60}")
    cursor_display = cursor if cursor else "(none — first page)"
    print(f"📄 PAGE {page_num}  (cursor={cursor_display}, limit={PAGE_SIZE})")
    print(f"{'═' * 60}")

    for event in data["events"]:
        print(f"  #{event['id']:>3}  {event['title'][:45]:<45}  [{event['category']}]")

    # Get the cursor for the next page
    pag = data["pagination"]
    cursor = pag["next_cursor"]
    print(f"\n  next_cursor={pag['next_cursor']}, has_more={pag['has_more']}")
    print()

🔖 Paginating with cursors (3 pages)

════════════════════════════════════════════════════════════
📄 PAGE 1  (cursor=(none — first page), limit=5)
════════════════════════════════════════════════════════════
  #  1  Jazz Night Vol. 1                              [tech]
  #  2  Comedy Show: Laughs Unlimited 2                [tech]
  #  3  Tech Conference 2024                           [sports]
  #  4  Classical Concert Series 4                     [tech]
  #  5  Rock Festival 2026 #5                          [sports]

  next_cursor=5, has_more=True

════════════════════════════════════════════════════════════
📄 PAGE 2  (cursor=5, limit=5)
════════════════════════════════════════════════════════════
  #  6  Jazz Night Vol. 6                              [sports]
  #  7  Comedy Show: Laughs Unlimited 7                [arts]
  #  8  Tech Conference 2026                           [comedy]
  #  9  Classical Concert Series 9                     [music]
  # 10  Rock Festival 2025 #10           

In [8]:
# ── Full Pagination Loop Pattern ─────────────────────────────────────────────
# This is how you'd fetch ALL pages in a real application.
# It's the standard "cursor loop" pattern you'll see everywhere.

all_events = []
cursor = None
page_count = 0

while True:
    # Build params
    params = {"limit": 10}
    if cursor is not None:
        params["cursor"] = cursor

    # Fetch one page
    data = api_get("/v2/events", **params)
    page_count += 1
    all_events.extend(data["events"])

    # Check if there are more pages
    if not data["pagination"]["has_more"]:
        break  # we've reached the last page!

    # Move the cursor forward
    cursor = data["pagination"]["next_cursor"]

print(f"✅ Fetched all {len(all_events)} events in {page_count} pages")
print(f"   First event: #{all_events[0]['id']} — {all_events[0]['title']}")
print(f"   Last event:  #{all_events[-1]['id']} — {all_events[-1]['title']}")

✅ Fetched all 52 events in 6 pages
   First event: #1 — Jazz Night Vol. 1
   Last event:  #60 — Nested Resource Demo


---

## ⚖️ Side-by-Side Comparison

| Feature              | Offset Pagination          | Cursor Pagination             |
|----------------------|---------------------------|-------------------------------|
| **Simplicity**       | ✅ Very simple             | ⚠️ Slightly more complex      |
| **Random access**    | ✅ Jump to page N          | ❌ Must go page by page        |
| **Performance**      | ❌ Degrades with offset    | ✅ Constant time (uses index)  |
| **Stability**        | ❌ Results shift on insert  | ✅ Stable results              |
| **Total count**      | ✅ Returns total           | ❌ No total (expensive to get) |
| **Implementation**   | ✅ Easy (OFFSET/LIMIT)     | ⚠️ Need ordered unique column  |
| **Best for**         | Admin panels, dashboards  | Feeds, infinite scroll, mobile|

Let's measure the actual performance difference:

In [9]:
# ── Performance Comparison ───────────────────────────────────────────────────
# Fetch all 50 events in batches of 10, using both pagination strategies.
# We time each approach to see which is faster.

BATCH_SIZE = 10

# ── Offset-based: fetch all pages ────────────────────────────────────────────
start = time.time()
offset_events = []
offset_val = 0

while True:
    data = api_get("/v1/events", offset=offset_val, limit=BATCH_SIZE)
    offset_events.extend(data["events"])
    if offset_val + BATCH_SIZE >= data["pagination"]["total"]:
        break
    offset_val += BATCH_SIZE

offset_time = time.time() - start

# ── Cursor-based: fetch all pages ────────────────────────────────────────────
start = time.time()
cursor_events = []
cursor = None

while True:
    params = {"limit": BATCH_SIZE}
    if cursor is not None:
        params["cursor"] = cursor
    data = api_get("/v2/events", **params)
    cursor_events.extend(data["events"])
    if not data["pagination"]["has_more"]:
        break
    cursor = data["pagination"]["next_cursor"]

cursor_time = time.time() - start

# ── Results ───────────────────────────────────────────────────────────────────
print("⏱️  Performance Comparison: Fetch all 50 events in batches of 10")
print("─" * 55)
print(f"  Offset-based (V1):  {offset_time:.4f}s  ({len(offset_events)} events)")
print(f"  Cursor-based (V2):  {cursor_time:.4f}s  ({len(cursor_events)} events)")
print("─" * 55)
print()
print("💡 With only 50 events, the difference is negligible.")
print("   But with millions of rows, offset pagination gets dramatically")
print("   slower on later pages, while cursor stays constant.")

⏱️  Performance Comparison: Fetch all 50 events in batches of 10
───────────────────────────────────────────────────────
  Offset-based (V1):  0.0279s  (52 events)
  Cursor-based (V2):  0.0256s  (52 events)
───────────────────────────────────────────────────────

💡 With only 50 events, the difference is negligible.
   But with millions of rows, offset pagination gets dramatically
   slower on later pages, while cursor stays constant.


## 🛠️ Implementing Pagination in SQL

Let's look at the raw SQL behind both approaches and compare their query plans.

In [10]:
# ── Raw SQL: Offset vs Cursor ────────────────────────────────────────────────
# Let's run both approaches directly against PostgreSQL and compare.

conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

# ── Offset-based SQL ─────────────────────────────────────────────────────────
print("📄 OFFSET-BASED SQL")
print("   SELECT * FROM events ORDER BY id OFFSET 20 LIMIT 5;\n")

offset_val = 20
limit_val = 5
cur.execute(
    "SELECT id, title, category FROM events ORDER BY id OFFSET %s LIMIT %s",
    (offset_val, limit_val)
)
rows = cur.fetchall()
for row in rows:
    print(f"  #{row[0]:>3}  {row[1][:45]:<45}  [{row[2]}]")

print(f"\n{'─' * 60}\n")

# ── Cursor-based SQL ──────────────────────────────────────────────────────────
print("🔖 CURSOR-BASED SQL")
print("   SELECT * FROM events WHERE id > 20 ORDER BY id LIMIT 5;\n")

cursor_val = 20
cur.execute(
    "SELECT id, title, category FROM events WHERE id > %s ORDER BY id LIMIT %s",
    (cursor_val, limit_val)
)
rows = cur.fetchall()
for row in rows:
    print(f"  #{row[0]:>3}  {row[1][:45]:<45}  [{row[2]}]")

print(f"\n{'─' * 60}")
print("\n💡 Both return the same rows! The difference is HOW the database finds them.")

# ── Compare Query Plans ──────────────────────────────────────────────────────
print(f"\n{'═' * 60}")
print("🔍 QUERY PLAN COMPARISON")
print(f"{'═' * 60}\n")

print("── Offset approach: EXPLAIN SELECT ... OFFSET 20 LIMIT 5 ──")
cur.execute("EXPLAIN SELECT * FROM events ORDER BY id OFFSET 20 LIMIT 5")
for row in cur.fetchall():
    print(f"  {row[0]}")

print("\n── Cursor approach: EXPLAIN SELECT ... WHERE id > 20 LIMIT 5 ──")
cur.execute("EXPLAIN SELECT * FROM events WHERE id > 20 ORDER BY id LIMIT 5")
for row in cur.fetchall():
    print(f"  {row[0]}")

print("\n💡 The cursor approach can use an Index Scan — it jumps directly")
print("   to the right row instead of scanning through all preceding rows.")

cur.close()
conn.close()

📄 OFFSET-BASED SQL
   SELECT * FROM events ORDER BY id OFFSET 20 LIMIT 5;

  # 21  Jazz Night Vol. 21                             [comedy]
  # 22  Comedy Show: Laughs Unlimited 22               [arts]
  # 23  Tech Conference 2026                           [sports]
  # 24  Classical Concert Series 24                    [music]
  # 25  Rock Festival 2025 #25                         [music]

────────────────────────────────────────────────────────────

🔖 CURSOR-BASED SQL
   SELECT * FROM events WHERE id > 20 ORDER BY id LIMIT 5;

  # 21  Jazz Night Vol. 21                             [comedy]
  # 22  Comedy Show: Laughs Unlimited 22               [arts]
  # 23  Tech Conference 2026                           [sports]
  # 24  Classical Concert Series 24                    [music]
  # 25  Rock Festival 2025 #25                         [music]

────────────────────────────────────────────────────────────

💡 Both return the same rows! The difference is HOW the database finds them.

═══════════

---

## 🧭 When to Use Which?

### Use Offset Pagination When:
- 📊 Your dataset is **small** (thousands, not millions)
- 🔢 Users need to **jump to a specific page** ("go to page 5")
- 🖥️ Building **admin dashboards** or **back-office tools**
- 📋 The data doesn't change often (static catalogs, reports)

### Use Cursor Pagination When:
- 📱 Building **infinite scroll** (Instagram, Twitter, Reddit)
- 🔄 Data changes frequently (**real-time feeds**, notifications)
- 📈 The dataset is **very large** (millions+ rows)
- 📲 Building **mobile apps** (where page numbers don't make sense)

### Real-World Examples

| Company    | Strategy                | Why                                   |
|------------|------------------------|---------------------------------------|
| **GitHub** | Cursor (GraphQL)       | Repos/issues can be massive           |
| **Stripe** | Cursor (`starting_after`) | Millions of transactions            |
| **Twitter**| Cursor (`max_id`)       | Timeline is a real-time feed          |
| **Google** | Offset (page numbers)  | Search results — you want "page 3"    |
| **Amazon** | Offset (page numbers)  | Product listings with page navigation |

### 💡 Interview Tip

In a system-design interview, **always mention pagination**. If the interviewer asks
you to design a social media feed or notification system, say:

> "For the feed endpoint, I'd use cursor-based pagination because the data is
> frequently updated and we need stable results as users scroll. The cursor would
> be the timestamp or ID of the last item returned."

This shows you understand real-world API design.

---

## 🎯 Key Takeaways

### 1. Always Paginate
Never return an unbounded result set. Every list endpoint should accept `limit`
and either `offset` or `cursor` parameters.

### 2. Offset Is Simple But Breaks at Scale
- ✅ Easy to implement: `OFFSET` + `LIMIT`
- ✅ Supports random page access
- ❌ Slow for large offsets (scans all skipped rows)
- ❌ Unstable when data changes between requests

### 3. Cursor Is Stable and Fast
- ✅ Constant performance regardless of position
- ✅ Stable results even when data changes
- ❌ No random page access (must go sequentially)
- ❌ Slightly more complex to implement

### 4. Choose Based on Your Use Case
- **Small data + page numbers** → Offset
- **Large data + infinite scroll** → Cursor

### 5. Interview Cheat Sheet
- Always mention pagination for any endpoint that returns a list
- Use cursor pagination for high-volume, real-time systems
- Know the SQL behind both approaches

---

**Next up:** Notebook 3 explores rate limiting — how to protect your API from being overwhelmed by too many requests. 🚦